# BruSafe · Notebook 01 — Module M1: expression, IPTG arm

*Section numbering follows the CAPSID framework document; this notebook implements Sections 2.1–2.6 and 2.11–2.20.*

**iGEM Tec-Chihuahua 2026 · CAPSID framework**

This notebook calibrates the expression arm of the pipeline, from IPTG addition to nascent coat protein (CP).

## Workflow

1. **M1 as a coupled ODE system.** The metabolic burden of CP synthesis enters as a feedback term inside the cascade.
2. **Literature-derived priors and a prior predictive simulation**, which give a yield forecast before any experiment.
3. **Identifiability audit**, including one exact structural degeneracy of the model.
4. **Synthetic data** standing in for experiment V-A, behind a single marked swap point.
5. **Calibration by sequential Monte Carlo (SMC)**, with a recovery check against known truth.
6. **Posterior predictive distribution**, including a held-out condition that is excluded from the fit.
7. **Export** of the joint posterior as the prior for the folding and aggregation module (M2).

---
## 1 · Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numba import njit
from scipy.integrate import solve_ivp
import pymc as pm, pytensor.tensor as pt, arviz as az
from pytensor.compile.ops import as_op

RNG = np.random.default_rng(2026)
plt.rcParams.update({"figure.dpi":110,"font.size":9,"axes.grid":True,"grid.alpha":.25,
                     "axes.spines.top":False,"axes.spines.right":False})
print("pymc", pm.__version__, "| arviz", az.__version__)

# ---- Figure style (team palette) -------------------------------------------
# The colours are sampled from the team wordmark. Figures refer to the names
# below instead of hex codes, so the whole set can be restyled from this cell.
# SF Pro is used where installed; matplotlib substitutes another font otherwise.
CREAM, PALE, STEEL = "#F4F3EF", "#CCD8E2", "#89A1C2"
SKY,   BLUE, MID   = "#4CA3DF", "#3E77E1", "#2C5BA5"
NAVY,  INK         = "#173779", "#122047"
from matplotlib.colors import LinearSegmentedColormap
BRAND_CMAP = LinearSegmentedColormap.from_list("tecchihuahua",
                                               [NAVY, MID, BLUE, SKY])
plt.rcParams.update({
    "font.family": ["SF Pro Text", "SF Pro Display", "SF Pro", "Helvetica Neue"],
    "mathtext.fontset": "custom", "mathtext.default": "regular",
    "mathtext.rm": "SF Pro Text", "mathtext.it": "SF Pro Text:italic",
    "mathtext.bf": "SF Pro Text:bold",
    "figure.facecolor": CREAM, "axes.facecolor": CREAM, "savefig.facecolor": CREAM,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": STEEL, "grid.alpha": 0.35,
    "legend.facecolor": CREAM, "legend.edgecolor": STEEL, "legend.framealpha": 0.9,
    "axes.prop_cycle": plt.cycler(color=[NAVY, BLUE, SKY, STEEL, MID, PALE]),
})


---
## 2 · The model

### 2.1 · State variables

| Symbol | Meaning | Units |
|---|---|---|
| `X` | biomass | gDCW L⁻¹ |
| `I` | intracellular inducer | μM |
| `R̃` | active T7 RNA polymerase | normalised 0–1 |
| `m̃` | CP transcript | normalised 0–1 |
| `P` | cumulative CP monomer | µM (cytoplasmic) |

### 2.2 · Equations

$$\phi_h = P/P_{ref}, \qquad \psi = \max\!\left(0,\ 1-\phi_h/\phi_{crit}\right)$$

$$\mu = \mu_{max}\max\!\left(0,1-\tfrac{X}{X_{max}}\right)\psi, \qquad \frac{dX}{dt}=\mu X$$

$$\frac{dI}{dt}=P_d(I_e-I)-\mu I, \qquad f(I)=\varepsilon+(1-\varepsilon)\frac{I^h}{K_I^h+I^h}$$

$$\frac{d\tilde R}{dt}=k_R(f-\tilde R),\qquad
\frac{d\tilde m}{dt}=k_m(\tilde R-\tilde m),\qquad
\frac{dP}{dt}=G\,\tilde m\,\psi-\mu P$$

(The code below adds a first-order loss term, `k_deg`, to the equation for `P`; the reason is given in §3.1.)

**(a) A single capacity factor ψ gates both growth and synthesis (M0).**
Under proteome-partitioning growth laws, expressing a non-functional protein displaces ribosomal
and metabolic sectors. The displacement that slows growth also limits further synthesis, so ψ
appears in both equations. As a result `P` saturates after growth arrests without an explicit cap.

**(b) The cascade is normalised and its gain is lumped into `G`.**
The physical chain (RNAP synthesis × plasmid copy number × transcription rate × translation rate)
is a product of four quantities that protein measurements cannot separate. R̃ and m̃ therefore
carry only the shape of the response, that is, the delay; all of the amplitude is in `G` (see §4).

**(c) Passive-only transport presumes a *lacY*⁻ strain.**
LacY is itself lac-controlled, so in *lacY*⁺ strains such as BL21(DE3) inducer uptake is
autocatalytic and the population becomes bimodal at sub-saturating IPTG. A deterministic mean then
corresponds to no real cell state, since part of the population is fully induced and part is off.
The model therefore applies to a *lacY*⁻ host such as Tuner(DE3), or to doses above roughly
100–200 µM. `I_e` is constant because IPTG is gratuitous. The model does not apply to lactose
induction, which additionally requires hydrolysis, allolactose formation and catabolite repression
(see notebook 02).

In [ ]:
M_CP  = 13.7      # kDa, MS2 coat protein monomer (Golmohammadi et al. 1993)
P_REF = 18000.0   # [M1.4] uM total cytoplasmic protein in CP-mass equivalents (~250 mg/mL)

# ---- Unit convention -------------------------------------------------------
# All internal quantities are in uM and minutes. mM appears only at the plotting
# and design interface, through the two helpers below; the two systems should
# not be mixed.
mM_to_uM = lambda x: np.asarray(x, float)*1e3
uM_to_mM = lambda x: np.asarray(x, float)/1e3
T_END    = 1440.0        # min. forecast horizon (24 h); an earlier version used 300 min (5 h)
# ----------------------------------------------------------------------------

# ---- Biomass and protein in physical units ---------------------------------
# OD600 and SDS-PAGE band volume are instrument readings, not concentrations,
# and band volumes in particular cannot be compared between gels. The model
# therefore works with biomass concentration (gDCW/L) and coat-protein
# concentration (mg/L); the raw readings enter only through the explicit
# calibrations defined below.
K_DCW = 0.40      # gDCW/L per OD600 unit (E. coli in rich medium: 0.3-0.5);
                  # should be calibrated for the strain in use.
F_DRY = 0.25      # dry mass / wet pellet mass; multiply wet weight by this to get gDCW/L.
od_to_gdcw  = lambda od:  np.asarray(od, float)*K_DCW
wet_to_gdcw = lambda gww: np.asarray(gww, float)*F_DRY
uM_to_mgL   = lambda c:   np.asarray(c, float)*M_CP/1000.0    # uM CP -> mg/L
mgL_to_uM   = lambda c:   np.asarray(c, float)*1000.0/M_CP
# ----------------------------------------------------------------------------

PNAMES = ["mu_max","Xmax","phi_crit","P_d","K_I","h","eps","k_R","k_m","G","k_deg"]

@njit(fastmath=True)
def _rhs(t, Y, p, I_e):
    n = I_e.size; out = np.empty(5*n)
    mu_max, Xmax, phi_crit, P_d = p[0], p[1], p[2], p[3]
    K_I, h, eps, k_R, k_m, G    = p[4], p[5], p[6], p[7], p[8], p[9]
    k_deg = p[10]                              # [M1.18] first-order CP loss
    Kh = K_I**h
    for i in range(n):
        X=Y[i]; I=Y[n+i]; R=Y[2*n+i]; m=Y[3*n+i]; P=Y[4*n+i]
        psi = 1.0 - (P/P_REF)/phi_crit          # [M1.2] proteome-partition growth law
        if psi < 0.0: psi = 0.0
        gx = 1.0 - X/Xmax                       # [M1.1] carrying capacity (phenomenological)
        if gx < 0.0: gx = 0.0
        mu = mu_max*gx*psi
        Ic = I if I > 0.0 else 0.0
        f  = eps + (1.0-eps)*Ic**h/(Kh+Ic**h)   # [M1.9] Hill de-repression  [M1.10] basal leak
        out[i]     = mu*X                       # [M1.1] logistic growth
        out[n+i]   = P_d*(I_e[i]-I) - mu*I      # [M1.5] passive uptake  [M1.8] dilution
        out[2*n+i] = k_R*(f-R)                  # [M1.11] RNAP relaxation
        out[3*n+i] = k_m*(R-m)                  # [M1.12] transcript turnover
        out[4*n+i] = G*m*psi - (mu + k_deg)*P     # [M1.16] gain  [M1.3] gating  [M1.18] loss
    return out

def simulate(theta, doses, t_eval, X0=0.16):
    # All doses are integrated as one augmented system, about 5x faster than looping.
    p = np.asarray(theta,float); I_e = np.asarray(doses,float)
    te = np.asarray(t_eval,float); n = I_e.size
    Y0 = np.concatenate([np.full(n,X0), np.zeros(4*n)])
    # rtol/atol are tighter than the 1e-6/1e-8 used earlier: over a 24-48 h horizon
    # the clip at psi = 0 is a kink, and the solver otherwise accumulates error across it.
    s = solve_ivp(_rhs,(0.0,float(te[-1])),Y0,t_eval=te,args=(p,I_e),
                  method="LSODA",rtol=1e-8,atol=1e-10)
    if (not s.success) or s.y.shape[1]!=te.size: return None
    Y=s.y
    return {"X":Y[0:n],"I":Y[n:2*n],"R":Y[2*n:3*n],"m":Y[3*n:4*n],"P":Y[4*n:5*n]}

def logspan(t_end, n=140, t0=2.0):
    # Log-spaced time grid. The transcript transient lasts minutes and the protein
    # plateau hours, so a linear grid would place almost all points on the plateau.
    return np.concatenate([[0.0], np.logspace(np.log10(t0), np.log10(t_end), n-1)])

#                mu_max Xmax  phi_c P_d  K_I   h   eps   k_R   k_m    G     k_deg
#                       gDCW/L
_th0 = np.array([0.021, 2.40, 0.25, 0.35, 30., 2., 0.004, 0.07, 0.30, 95., 0.0035])
simulate(_th0, np.array([25.]), np.array([0.,60.]))   # triggers JIT compilation
import time
_t=time.time()
for _ in range(200): simulate(_th0, np.array([10.,25.,50.,200.]), np.linspace(0,T_END,6))
print(f"forward model: {(time.time()-_t)/200*1000:.2f} ms per evaluation")

### 2.3 · Term-by-term provenance

Every term in `_rhs` carries a tag `[Mx.y]` that indexes the table below. The *Evidence* column
records how each term is supported:

| Class | Meaning |
|---|---|
| **Direct** | The cited work reports this functional form or numerical value |
| **Adapted** | The literature establishes the mechanism; the specific functional form is ours |
| **Assumption** | A modelling choice with no literature claim behind it, justified by argument rather than citation |
| **Reparam.** | A mathematical restatement, not a physical claim; it carries no empirical content |

Classifying a term as an assumption is a statement about its status and not a weakness of the
model. The error to avoid is presenting an assumption as a direct result.

In [ ]:
CITATIONS = pd.DataFrame([
 ("M1.1","dX/dt = mu*X; (1 - X/Xmax)","Logistic carrying capacity stands in for unmodelled nutrient depletion",
  "Assumption","Monod 1949 Annu Rev Microbiol 3:371 gives the mechanistic (substrate-explicit) alternative"),
 ("M1.2","psi = 1 - phi_h/phi_crit","Growth rate falls ~linearly with the mass fraction of non-functional protein",
  "Direct","Scott et al. 2010 Science 330:1099; Klumpp, Zhang & Hwa 2009 Cell 139:1366"),
 ("M1.3","psi also multiplies synthesis","Growth and heterologous expression draw on one shared ribosomal pool",
  "Adapted","Weisse et al. 2015 PNAS 112:E1038. The shared-factor FORM is ours -- see note below"),
 ("M1.4","P_REF = 18000 uM","Cytoplasmic protein ~200-300 mg/mL, expressed in CP-mass equivalents",
  "Direct","Bremer & Dennis 2008 EcoSal Plus; Milo et al. 2010 NAR 38:D750 (BioNumbers)"),
 ("M1.5","P_d*(I_e - I)","IPTG enters independently of LacY at higher concentrations",
  "Direct","Marbach & Bettenbrock 2012 J Biotechnol 157:82"),
 ("M1.6","LacY term OMITTED","lacY-minus strain removes the saturable carrier and its positive feedback",
  "Direct","Marbach & Bettenbrock 2012 (bimodality at low IPTG in lacY+); Ozbudak et al. 2004 Nature 427:737"),
 ("M1.7","I_e held constant","IPTG is gratuitous -- not metabolised, so external pool is not depleted",
  "Direct","Marbach & Bettenbrock 2012. FAILS for lactose -- see M1.7 note"),
 ("M1.8","-mu*I dilution","Volume expansion dilutes every intracellular species",
  "Reparam.","Standard mass balance; no empirical claim"),
 ("M1.9","Hill f(I), h ~ 2","Cooperative inducer binding to LacI; two IPTG sites per dimer",
  "Adapted","Lewis et al. 1996 Science 271:1247 (structure); Yildirim & Mackey 2003 Biophys J 84:2841 (n=2 in model)"),
 ("M1.10","eps (basal leak)","lacUV5 in the DE3 lysogen is leaky; magnitude is construct-dependent",
  "Direct","Studier & Moffatt 1986 JMB 189:113; Dubendorff & Studier 1991 JMB 219:45"),
 ("M1.11","k_R*(f - R)","First-order relaxation lumps RNAP synthesis, folding and dilution into one constant",
  "Assumption","No literature reports this lumped constant. It is a time-scale placeholder."),
 ("M1.12","k_m*(R - m)","Transcript turnover; E. coli median mRNA half-life ~5 min",
  "Direct","Bernstein et al. 2002 PNAS 99:9697 (median 5 min); Selinger et al. 2003 Genome Res 13:216 (6.8 min)"),
 ("M1.13","elongation NOT modelled","T7 RNAP elongates ~230 nt/s at 37 C, ~5x E. coli RNAP -- transcript completion is fast vs all other time scales",
  "Direct","Iost, Guillerez & Dreyfus 1992 J Bacteriol 174:619; Vogel & Jensen 1994 J Bacteriol 176:2807 (E. coli 28-89 nt/s)"),
 ("M1.14","shared RNAP between cassettes","Two T7 promoters compete for one polymerase pool",
  "Adapted","Klumpp & Hwa 2008 PNAS 105:20245 (RNAP partitioning). Not implemented in M1; see §2 note (b)"),
 ("M1.15","CP represses cargo translation","MS2 CP dimer binding the operator blocks downstream translation",
  "Direct","Bernardi & Spahr 1972 PNAS 69:3033; Peabody 1993 EMBO J 12:595. Include ONLY if the pac array is 5' of the epitope ORF"),
 ("M1.16","G (lumped gain)","Product of RNAP level, plasmid copy no., k_tx and k_tl -- not separable from protein data",
  "Reparam.","No citation possible or appropriate. See identifiability audit, §4.2"),
 ("M1.17","ribosome saturation","Translation capacity saturates as transcript load rises",
  "Adapted","Scott et al. 2010; Klumpp et al. 2009. Absorbed into G at this level of detail"),
 ("M1.18","-(mu + k_deg)*P","First-order CP loss lumping proteolysis, inclusion-body partition and leakage. "
  "Without it dP/dt loses every sink once mu -> 0, and P clamps at phi_crit*P_REF for all doses",
  "Assumption","Maurizi 1992 Experientia 48:178 gives ClpP/Lon turnover ranges; no MS2-CP value exists. "
  "Measure it directly with a translation-halt chase -- see §3.1"),
 ("M1.19","log-normal gel error","Band-intensity error is multiplicative at high signal, additive at low",
  "Adapted","Rocke & Durbin 2001 J Comput Biol 8:557 (two-component intensity error model)"),
 ("M1.20","per-gel alpha","Loading, transfer and exposure vary between gels: a batch effect",
  "Adapted","Gelman & Hill 2007, Data Analysis Using Regression and Multilevel Models, ch. 12"),
 ("M1.21","discrepancy term","Absorbs structural model error instead of distorting kinetic parameters",
  "Direct","Kennedy & O'Hagan 2001 JRSS-B 63:425. NOT yet implemented -- add if §10 coverage fails"),
 ("M1.22","SMC sampler","Tempered sequential importance sampling; gradient-free, handles multimodality",
  "Direct","Del Moral, Doucet & Jasra 2006 JRSS-B 68:411"),
 ("M1.23","identifiability audit","Structural and practical identifiability must precede inference",
  "Direct","Raue et al. 2009 Bioinformatics 25:1923"),
 ("M1.24","k_R <-> k_m symmetry","Cascade transfer function is symmetric in its two poles",
  "Reparam.","Derived in §4.1, not cited -- it is a provable property of the model, verified numerically"),
], columns=["tag","term","basis","evidence","reference"])

pd.set_option("display.max_colwidth", 110)
print(CITATIONS.evidence.value_counts().to_string(), "\n")
CITATIONS

> **Note on M1.3.** That growth rate declines with heterologous expression is well established
> (M1.2). That the same factor also scales further synthesis is a modelling choice of ours. It is
> motivated by shared ribosome allocation, but it has not been published in this form. It matters
> because it is what makes `P` saturate.
>
> The term is treated as an assumption. It could be tested by measuring ribosome content (or rRNA)
> across the induction series and checking whether it tracks ψ.

In [ ]:
def mass_balance_residual(theta, doses, t_end=None):
    # Integrity check: CP synthesised = CP accumulated + dilution + degradation losses.
    t_end = T_END if t_end is None else t_end
    te = np.linspace(0,t_end,1201); o = simulate(theta,doses,te); p = np.asarray(theta,float)
    psi = np.clip(1.0-(o["P"]/P_REF)/p[2],0,None)
    mu  = p[0]*np.clip(1.0-o["X"]/p[1],0,None)*psi
    made = np.trapezoid(p[9]*o["m"]*psi, te, axis=1)
    lost = np.trapezoid((mu + p[10])*o["P"], te, axis=1)   # [M1.18] loss now in the ledger
    return np.abs(o["P"][:,-1]-(made-lost))/np.maximum(made,1e-9)

# checked at the LONG horizon and across the full uM->mM span, not just 300 min
for _te in (300.0, T_END, 2880.0):
    r = mass_balance_residual(_th0, np.array([10.,50.,200.,1000.,10000.]), _te)
    print(f"t_end={_te/60:5.1f} h  max relative residual = {r.max():.2e}")
    assert r.max() < 1e-3, "MASS BALANCE VIOLATED -- stoichiometry bug"
print("PASS")

---
## 3 · Literature priors and the pre-experimental forecast

The priors are deliberately wider than the literature point estimates would suggest. A prior that
is too tight conflicts with the data and moves the disagreement into the discrepancy term.

In [ ]:
PRIORS = pd.DataFrame([
 ("mu_max","TruncN",0.021,0.005,"M1.1","Doubling ~33 min, rich medium 37 C",
  "Bremer & Dennis 2008 EcoSal Plus","Direct"),
 ("Xmax","TruncN",2.40,0.60,"M1.1","Shake-flask carrying capacity, gDCW/L (= 6 OD600 x K_DCW)",
  "Measured with our own negative control","Assumption"),
 ("phi_crit","TruncN",0.25,0.08,"M1.2","Max heterologous proteome fraction; strong T7 reaches 20-30%",
  "Studier & Moffatt 1986 JMB 189:113; Scott et al. 2010 Science 330:1099","Direct"),
 ("P_d","LogN",0.35,0.6,"M1.5","Passive IPTG entry, equilibrates within minutes",
  "Marbach & Bettenbrock 2012 J Biotechnol 157:82","Adapted"),
 ("K_I","LogN",30.0,0.7,"M1.9","Half-maximal IPTG, lacY-minus background",
  "Marbach & Bettenbrock 2012 J Biotechnol 157:82","Direct"),
 ("h","TruncN",2.0,0.5,"M1.9","Two inducer sites per LacI dimer",
  "Lewis et al. 1996 Science 271:1247; Yildirim & Mackey 2003 Biophys J 84:2841","Adapted"),
 ("eps","LogN",0.004,0.8,"M1.10","lacUV5 basal leak in DE3; strongly construct-dependent",
  "Studier & Moffatt 1986; Dubendorff & Studier 1991 JMB 219:45","Adapted"),
 ("k_R","LogN",0.07,0.6,"M1.11","T7 RNAP accumulation, ~15 min time constant",
  "Order-of-magnitude from Studier & Moffatt 1986; the lumped constant is not itself reported","Assumption"),
 ("k_m","LogN",0.30,0.6,"M1.12","Transcript turnover; median E. coli mRNA half-life ~5 min",
  "Bernstein et al. 2002 PNAS 99:9697; Selinger et al. 2003 Genome Res 13:216","Direct"),
 ("G","LogN",95.0,0.8,"M1.16","LUMPED GAIN: RNAP x copy no. x k_tx x k_tl",
  "No citation possible -- centred to give ~20% proteome fraction at saturation","Reparam."),
 ("k_deg","LogN",0.0035,0.9,"M1.18","Lumped CP loss: proteolysis + IB partition + leakage. "
  "t_1/2 ~ 3 h. UNCONSTRAINED by a 5 h experiment -- see the horizon audit in §3.1",
  "Maurizi 1992 Experientia 48:178 (ClpP/Lon ranges); no MS2-CP-specific value exists","Assumption"),
], columns=["param","dist","loc","scale","tag","basis","reference","evidence"])
BOUNDS = {"mu_max":(0.005,0.045),"Xmax":(0.8,4.8),"phi_crit":(0.05,0.50),"h":(1.,4.)}
pd.set_option("display.max_colwidth",78)
print("Prior provenance by evidence class:")
print(PRIORS.evidence.value_counts().to_string(), "\n")
PRIORS

In [ ]:
def sample_prior(n, rng=RNG):
    out={}
    for _,r in PRIORS.iterrows():
        if r["dist"]=="LogN":
            out[r["param"]] = rng.lognormal(np.log(r["loc"]), r["scale"], n)
        else:
            lo,hi = BOUNDS[r["param"]]; v = rng.normal(r["loc"], r["scale"], n)
            while np.any(bad:=(v<lo)|(v>hi)): v[bad]=rng.normal(r["loc"],r["scale"],bad.sum())
            out[r["param"]]=v
    return pd.DataFrame(out)[PNAMES]

# ---- Dose design -----------------------------------------------------------
# The calibration ladder must straddle K_I (prior median 30 uM), otherwise K_I
# and h are not identifiable (see the saturation audit two cells down). The
# 1 mM rung ties the ladder to the standard induction protocol; it carries almost
# no information about K_I, but it is the condition most commonly used.
DOSES = np.array([5., 15., 50., 150., 500., 1000.])   # uM IPTG (experiment V-A)

# Sampling times: roughly log-spaced and extended past the plateau. The two late
# points are the only data that can constrain k_deg.
TOBS  = np.array([0., 20., 45., 90., 180., 300., 720., 1440.])   # min post-induction

PHI_CYT = 1.8e-3          # gCP per gDCW per uM of CP  (cytoplasmic volume factor)
yield_mgL = lambda P_uM, X_gdcw: P_uM*1e-6*M_CP*1e3*PHI_CYT*X_gdcw*1e3

# Forecast scan: 1 uM to 10 mM, log-spaced. Linear spacing over 0-10 mM would put
# about 90% of the points inside the saturated plateau.
dscan = np.logspace(0, 4, 30)                          # uM
t_fine = logspan(T_END, 160)

prior_df = sample_prior(400)
Pp,ODp = [],[]
for _,row in prior_df.iterrows():
    o = simulate(row.values, DOSES, t_fine)
    if o is not None: Pp.append(o["P"]); ODp.append(o["X"])
Pp,ODp = np.array(Pp),np.array(ODp); Ypp = yield_mgL(Pp,ODp)
print(f"{Pp.shape[0]}/400 prior draws integrated over {T_END/60:.0f} h")
print("doses (mM):", np.array2string(uM_to_mM(DOSES), precision=3))

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(12.5,3.5)); qs=[5,25,50,75,95]
for j,d in enumerate(DOSES):
    q=np.percentile(Ypp[:,j,:],qs,axis=0); c=BRAND_CMAP(j/(len(DOSES)-1))
    ax[0].fill_between(t_fine/60,q[0],q[4],alpha=.13,color=c)
    ax[0].plot(t_fine/60,q[2],color=c,lw=1.8,label=f"{uM_to_mM(d):.3g}")
    qo=np.percentile(ODp[:,j,:],qs,axis=0)
    ax[1].fill_between(t_fine/60,qo[0],qo[4],alpha=.13,color=c); ax[1].plot(t_fine/60,qo[2],color=c,lw=1.8)
ax[0].set(xlabel="time post-induction (h)",ylabel="CP yield (mg/L)",title="Prior predictive: yield")
ax[0].legend(fontsize=7,title="IPTG (mM)")
ax[1].set(xlabel="time post-induction (h)",ylabel="biomass (gDCW L$^{-1}$)",title="Prior predictive: growth (burden)")

Yd=[]
for _,row in prior_df.iloc[:200].iterrows():
    o=simulate(row.values,dscan,np.array([0.,T_END]))
    if o is not None: Yd.append(yield_mgL(o["P"][:,-1],o["X"][:,-1]))
Yd=np.array(Yd); qd=np.percentile(Yd,qs,axis=0)
ax[2].fill_between(uM_to_mM(dscan),qd[0],qd[4],alpha=.18,color=MID)
ax[2].fill_between(uM_to_mM(dscan),qd[1],qd[3],alpha=.28,color=MID)
ax[2].plot(uM_to_mM(dscan),qd[2],color=MID,lw=2)
for d in DOSES: ax[2].axvline(uM_to_mM(d),color=INK,lw=.6,alpha=.35)
ax[2].set(xscale="log",xlabel="IPTG (mM)",ylabel=f"CP yield at {T_END/60:.0f} h (mg/L)",
          title="Predicted dose-response")
plt.tight_layout(); plt.show()
print(f"Prior-predicted optimum ~{uM_to_mM(dscan[np.argmax(qd[2])]):.3g} mM "
      f"({qd[2].max():.0f} mg/L) vs {qd[2][-1]:.0f} mg/L at 10 mM")

### 3.1 · Saturation and horizon audit

Two failure modes appear once the forecast is extended to the mM range and to 24 h. Neither raises
an error: the code runs, the plots look reasonable, and the numbers are wrong. The cell below makes
both visible.

In [ ]:
# ---- A. how much signal survives above K_I? --------------------------------
_probe = np.array([250., 500., 1000., 2000., 5000., 10000.])   # uM
_P = simulate(_th0, _probe, np.array([0., T_END]))["P"][:,-1]
_hi = _P[_probe >= 1000.]
_sig = (_hi.max()-_hi.min())/_hi.mean()
print("A. dose separation in the mM range")
for d,pp in zip(_probe,_P): print(f"     {uM_to_mM(d):6.2f} mM -> P = {pp:9.3f} uM")
print(f"   relative spread over 1-10 mM : {100*_sig:.4f} %")
print(f"   densitometry noise (sig_cp)  : 12 %")
print(f"   -> SNR = {_sig/0.12:.1e}. Doses above ~{5*_th0[4]:.0f} uM are one condition, not several.\n")

# ---- B. is K_I identifiable from an all-mM ladder? -------------------------
print("B. K_I identifiability, all-mM design vs. straddling design")
for _KI in [5., 30., 150., 500.]:
    _th = _th0.copy(); _th[4] = _KI
    _a = simulate(_th, np.array([1e3,5e3,1e4]), np.array([0.,T_END]))["P"][:,-1]
    _b = simulate(_th, DOSES, np.array([0.,T_END]))["P"][:,-1]
    print(f"   K_I={_KI:6.1f}  all-mM: {np.array2string(_a,precision=1)}"
          f"   straddling: {np.array2string(_b,precision=1)}")
print("   all-mM rows are near-identical -> flat posterior, no shrinkage vs prior.\n")

# ---- C. does the long-horizon forecast still depend on dose? ---------------
print(f"C. horizon audit (k_deg = {_th0[10]:.4f} /min, t_1/2 = {np.log(2)/_th0[10]/60:.1f} h)")
_ds = np.array([5.,15.,50.,150.,500.,1000.,10000.])
for _te in [300., 720., 1440., 2880.]:
    for _kP,_lbl in [(0.0,"k_deg=0  "), (_th0[10],"k_deg>0  ")]:
        _th = _th0.copy(); _th[10] = _kP
        _p = simulate(_th,_ds,np.array([0.,_te]))["P"][:,-1]
        print(f"   {_lbl} t={_te/60:5.1f} h  spread across doses = "
              f"{100*(_p.max()-_p.min())/_p.mean():6.1f} %"
              f"   ceiling hit: {np.sum(_p > 0.995*_th[2]*P_REF)}/{_ds.size}")
print("   With k_deg=0 the only brake on P is psi, so every dose converges on")
print(f"   phi_crit*P_REF = {_th0[2]*P_REF:.0f} uM and the dose-response flattens to noise.")

**Interpretation.**

*Checks A and B* concern experimental design. Doses in the mM range behave as a single condition.
Calibrating only there returns the prior on `K_I` and `h`, with an acceptable-looking ESS and no
shrinkage: the sampler works, but the data contain no information about those parameters.
Forecasting at mM doses is appropriate, since that is what the wet lab runs; calibrating only at
mM doses is not. `DOSES` straddles `K_I` for this reason.

*Check C* reveals a structural property. In the original equation `dP/dt = G·m·ψ − µP`, the growth
rate is itself `µ = µ_max·g(X)·ψ`. Once the culture reaches carrying capacity, `g(X) → 0` and
therefore `µ → 0`, and the protein equation loses its only loss term. `P` then rises until `ψ = 0`
and stops at `φ_crit·P_REF`, which is the same value for every dose. Over 5 h this is not visible;
over 24 h it determines the result, with 289 of 300 prior draws within 0.5% of that ceiling at 1 mM.

`k_deg` (M1.18) is the minimal correction. It is a first-order loss that lumps proteolysis,
inclusion-body partition and leakage, and it gives `P` a dose-dependent steady state,
`P_ss = G·m̃·ψ/(µ + k_deg)`, instead of a clamp.

> **Limitation.** `k_deg` is now the weakest element of the long-horizon forecast. No
> MS2-CP-specific value exists in the literature, and a 5 h time course cannot constrain it,
> because every trajectory is still rising at 300 min. Yet it determines the headline
> recommendation: at `k_deg = 0` the predicted optimum shifts *down* toward 15 µM as the horizon
> grows, whereas at `k_deg = 0.0035` it shifts *up*. The same data can therefore support opposite
> recommendations, depending on a parameter that has not been measured.

The remedy is inexpensive and follows the model → experiment → model loop: include the 12 h and
24 h time points already listed in `TOBS`. Two additional gel lanes per condition turn `k_deg`
from an assumption into a measurement. A more direct option is an arm in which translation is
halted at 300 min with an inhibitor such as chloramphenicol. With synthesis switched off, the decay
of the CP band is `k_deg`, read directly from the gel without ODE fitting. (Rifampicin is not
suitable for this purpose, because T7 RNA polymerase is rifampicin-resistant.)

### 3.2 · An emergent prediction

The right-hand panel is not monotonic. Volumetric yield peaks at an intermediate IPTG dose and then
declines, because beyond a threshold the burden term suppresses biomass faster than per-cell
expression increases. Protein per cell keeps rising; protein per litre does not.

This optimum is also computed before folding is considered. The partition model of M2 predicts a
soluble fraction that falls as the synthesis rate rises, so the optimum for *soluble* yield lies at
a lower dose still. Two independent mechanisms acting in the same direction make this a coherent and
testable design claim.

---
## 4 · Identifiability audit

### 4.1 · An exact symmetry in the cascade

The transfer function of the cascade from `f` to `m̃` is

$$H(s)=\frac{k_R}{s+k_R}\cdot\frac{k_m}{s+k_m}$$

which is exactly symmetric under `k_R ↔ k_m`: swapping the two constants gives an identical protein
trajectory.

In [ ]:
th_a = _th0.copy(); th_b = _th0.copy(); th_b[7],th_b[8] = _th0[8],_th0[7]
a,b = simulate(th_a,DOSES,t_fine), simulate(th_b,DOSES,t_fine)
print(f"max|P_a - P_b|/max(P) = {np.abs(a['P']-b['P']).max()/a['P'].max():.3e}  -> exchangeable\n")

fig,ax=plt.subplots(1,2,figsize=(9,3.2))
ax[0].plot(t_fine,a["P"][2],lw=3.5,alpha=.45,label="$k_R$=0.07, $k_m$=0.30")
ax[0].plot(t_fine,b["P"][2],"--",lw=1.6,color=INK,label="$k_R$=0.30, $k_m$=0.07")
ax[0].set(xlabel="time (min)",ylabel="CP ($\\mu$M)",title="Observable: identical")
ax[0].legend(fontsize=7)
ax[1].plot(t_fine,a["R"][2],lw=2,label="$\\tilde R$ original")
ax[1].plot(t_fine,b["R"][2],"--",lw=2,label="$\\tilde R$ swapped")
ax[1].set(xlabel="time (min)",ylabel="normalised",title="Swapped states: different")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

The symmetry is broken by convention. The sampler draws an ordered pair, a slow constant `k_slow`
and a ratio `ρ ∈ (0,1)` with `k_fast = k_slow/ρ`, and the slower constant is assigned to R̃ as a
labelling convention.

### 4.2 · Remaining degeneracies

| Group | Status | Resolution |
|---|---|---|
| `α_R·k_tx·N_p·k_tl` | Exactly non-identifiable | Lumped into `G` |
| `k_R ↔ k_m` | Exact symmetry | Ordering constraint (above) |
| `P_d` | Weakly identified | Transport equilibrates within minutes; needs samples at 5, 10 and 15 min |
| `eps` | Weakly identified | Needs an uninduced arm carried to late times |

---
## 5 · Data

> ### ⇄ SWAP POINT: the only cell to change when real data arrive
>
> Required arrays:
> - `X_obs`: `(n_doses, n_times)`, biomass in gDCW L⁻¹ (from OD₆₀₀ × `K_DCW`, or from wet pellet weight × `F_DRY`)
> - `CP_obs`: `(n_doses, n_times)`, densitometry in **arbitrary units**
> - `GEL_ID`: length `n_doses`, the physical gel on which each condition was run
>
> Equal OD-units should be loaded in every lane and the gel identity recorded, because the model
> fits a separate scale for each gel (§6). Densitometry should not be pre-normalised; raw
> integrated band volumes are expected.
>
> **Fractionation (relevant from M2 onward).** Cells are lysed in non-denaturing buffer and
> centrifuged, and SDS loading buffer is added afterwards, separately to the supernatant and to the
> resuspended pellet.

In [ ]:
USE_SYNTHETIC = True          # set to False once the V-A data are available
THETA_TRUE = _th0.copy()

if USE_SYNTHETIC:
    # Two gels, three conditions each. GEL_ID must be defined before it is used
    # to index ALPHA_TRUE, and must cover every dose in DOSES.
    GEL_ID     = np.array([0,0,0,1,1,1])
    ALPHA_TRUE = np.array([1.0,0.78])       # standard-curve slope per gel, unknown to the model
    truth  = simulate(THETA_TRUE, DOSES, TOBS)
    # biomass in gDCW/L with 5% relative error (balance + sampling volume)
    X_obs  = truth["X"]*np.exp(RNG.normal(0,0.05,truth["X"].shape))
    # coat protein in mg/L: densitometry read against a purified standard curve,
    # 12% relative error, one slope per gel
    CP_obs = (ALPHA_TRUE[GEL_ID][:,None]*uM_to_mgL(truth["P"])+0.05)*np.exp(RNG.normal(0,0.12,truth["P"].shape))
    print("SYNTHETIC data. Ground truth known -> recovery is checkable in §8.")
else:
    # Expected columns: dose_uM, time_min, gel_id, and EITHER biomass_gdcw_L or
    # OD600 or wet_weight_g_L, plus CP_mg_L (densitometry against a standard curve).
    df = pd.read_csv("VA_timecourse.csv")
    dr = np.sort(df.dose_uM.unique()); tr = np.sort(df.time_min.unique())
    piv = lambda c: df.pivot_table(index="dose_uM",columns="time_min",values=c).loc[dr,tr].values
    if   "biomass_gdcw_L" in df: X_obs = piv("biomass_gdcw_L")
    elif "wet_weight_g_L" in df: X_obs = wet_to_gdcw(piv("wet_weight_g_L"))
    else:                        X_obs = od_to_gdcw(piv("OD600"))
    CP_obs = piv("CP_mg_L")
    GEL_ID = df.groupby("dose_uM").gel_id.first().loc[dr].values.astype(int)
    DOSES, TOBS = dr, tr
    print(f"Real data loaded: {X_obs.shape[0]} doses x {X_obs.shape[1]} timepoints")
    print(f"  biomass {X_obs.min():.2f}-{X_obs.max():.2f} gDCW/L | CP {CP_obs.min():.1f}-{CP_obs.max():.1f} mg/L")

N_GEL = int(GEL_ID.max())+1; LOG_CP = np.log(CP_obs)
print("doses",DOSES,"| times",TOBS,"|",N_GEL,"gel(s)")
print(f"biomass {X_obs.min():.2f}-{X_obs.max():.2f} gDCW/L "
      f"(= {X_obs.min()/F_DRY:.1f}-{X_obs.max()/F_DRY:.1f} g wet weight/L) | "
      f"CP {CP_obs.min():.2f}-{CP_obs.max():.1f} mg/L")

---
## 6 · Observation model

**Per-gel scale `α_k`.** Densitometry is in arbitrary units, and loading, transfer and exposure
vary between gels. A single global scale would force the sampler to distort the kinetic parameters
to absorb a purely technical artefact. The scale of gel 0 is fixed at 1 to resolve the degeneracy
between scale and the gain `G`.

**Log-normal error.** Band-intensity error scales with intensity, so a fixed relative error is the
appropriate noise model.

**Biomass noise floor.** The absolute error of a biomass measurement does not vanish as the culture
thins, so a small additive term is included.

In [ ]:
@as_op(itypes=[pt.dvector], otypes=[pt.dmatrix, pt.dmatrix])
def forward_op(theta):
    o = simulate(theta, DOSES, TOBS)
    if o is None:
        bad = np.full((DOSES.size,TOBS.size), 1e6)
        return bad, bad
    # second output is log of the PREDICTED CP CONCENTRATION in mg/L, so the
    # observation model works on the same quantity the gel reports
    return o["X"], np.log(o["P"]*M_CP/1000.0 + 0.05)

def build_model():
    with pm.Model() as M:
        mu_max = pm.TruncatedNormal("mu_max",0.021,0.005,lower=0.005,upper=0.045)
        Xmax   = pm.TruncatedNormal("Xmax",6.0,1.5,lower=2.0,upper=12.0)
        phic   = pm.TruncatedNormal("phi_crit",0.25,0.08,lower=0.05,upper=0.50)
        P_d    = pm.LogNormal("P_d",np.log(0.35),0.6)
        K_I    = pm.LogNormal("K_I",np.log(30.0),0.7)
        h      = pm.TruncatedNormal("h",2.0,0.5,lower=1.0,upper=4.0)
        eps    = pm.LogNormal("eps",np.log(0.004),0.8)
        G      = pm.LogNormal("G",np.log(95.0),0.8)
        k_deg    = pm.LogNormal("k_deg",np.log(0.0035),0.9)   # [M1.18] CP loss

        # ordered cascade constants -- breaks the exact k_R <-> k_m symmetry (§4.1)
        k_slow = pm.LogNormal("k_slow",np.log(0.07),0.6)
        rho    = pm.Beta("rho",2.0,2.0)
        k_fast = pm.Deterministic("k_fast", k_slow/rho)

        # per-gel scaling; gel 0 pinned at 1 to resolve degeneracy with G
        if N_GEL > 1:
            la_free = pm.Normal("log_alpha_free",0.0,0.3,shape=N_GEL-1)
            log_alpha = pm.Deterministic("log_alpha",
                            pt.concatenate([pt.zeros(1), la_free]))
        else:
            log_alpha = pm.Deterministic("log_alpha", pt.zeros(1))

        sig_X  = pm.HalfNormal("sig_X",0.08)   # relative error on gDCW/L
        sig_cp = pm.HalfNormal("sig_cp",0.20)  # relative error on mg/L

        theta = pt.stack([mu_max,Xmax,phic,P_d,K_I,h,eps,k_slow,k_fast,G,k_deg])
        X_m, logcp_m = forward_op(theta)
        pm.Normal("X_lik", mu=X_m, sigma=sig_X*X_m+0.008, observed=X_obs)
        pm.Normal("cp_lik", mu=log_alpha[GEL_ID][:,None]+logcp_m,
                  sigma=sig_cp, observed=LOG_CP)
    return M

model = build_model(); print("model built")

---
## 7 · Sequential Monte Carlo sampling

The forward model is a `scipy` ODE solve wrapped in a `pytensor` Op, so gradients are unavailable
and NUTS cannot be used. SMC is gradient-free, copes with residual multimodality, tempers through
the likelihood instead of random-walking across it, and returns an estimate of the marginal
likelihood.

In [ ]:
# FAST_MODE gives a usable posterior in a few minutes on a laptop.
# Set to False for the numbers reported in the write-up.
FAST_MODE = False
DRAWS, CHAINS = (400, 4) if FAST_MODE else (2000, 4)
print(f"SMC: {DRAWS} draws x {CHAINS} chains"
      f"{'  [FAST_MODE -- re-run with FAST_MODE=False before reporting]' if FAST_MODE else ''}")

import time; _t0 = time.time()
with model:
    idata = pm.sample_smc(draws=DRAWS, chains=CHAINS, random_seed=2026, progressbar=False)
print(f"done in {time.time()-_t0:.0f} s |", dict(idata.posterior.sizes))

---
## 8 · Diagnostics and recovery

In [ ]:
VARS=["mu_max","Xmax","phi_crit","P_d","K_I","h","eps","k_slow","k_fast","G","k_deg","sig_X","sig_cp"]
summ = az.summary(idata, var_names=VARS); display(summ)
bad = summ[summ.r_hat > 1.01]
print(("\n!! r_hat>1.01, do NOT trust: "+", ".join(bad.index)) if len(bad)
      else "\nAll r_hat <= 1.01")
low = summ[summ.ess_bulk < 200]
print(("!! ESS<200: "+", ".join(low.index)) if len(low) else "All ESS >= 200")

In [ ]:
if USE_SYNTHETIC:
    tm = dict(zip(PNAMES, THETA_TRUE))
    tm["k_slow"], tm["k_fast"] = min(THETA_TRUE[7],THETA_TRUE[8]), max(THETA_TRUE[7],THETA_TRUE[8])
    rows=[]
    for v in ["mu_max","Xmax","phi_crit","P_d","K_I","h","eps","k_slow","k_fast","G","k_deg"]:
        s = idata.posterior[v].values.ravel(); lo,hi = np.percentile(s,[3,97])
        rows.append({"param":v,"truth":tm[v],"median":np.median(s),
                     "lo94":lo,"hi94":hi,"recovered":bool(lo<=tm[v]<=hi)})
    rec = pd.DataFrame(rows); display(rec.round(4))
    print(f"\n{rec.recovered.sum()}/{len(rec)} recovered within the 94% credible interval.")
    print("~1 miss in 10 is expected by chance. Two or more -> diagnose before trusting real data.")

In [ ]:
# Hand-written instead of az.plot_forest / az.plot_pair: the arviz plotting API
# changed substantially at 1.0, and these functions run on any version.
def kde1d(x, grid):
    x = np.asarray(x); bw = 1.06*np.std(x)*x.size**(-0.2) + 1e-12
    z = (grid[:,None]-x[None,:])/bw
    return np.exp(-0.5*z**2).sum(1)/(x.size*bw*np.sqrt(2*np.pi))

_pri = sample_prior(4000)
SHOW = ["K_I","h","phi_crit","G","k_slow","k_deg","mu_max","eps"]
_map = {"k_slow":"k_R"}   # prior for the ordered constant comes from the k_R prior
fig,axes = plt.subplots(2,4,figsize=(12,5.2))
for ax,v in zip(axes.ravel(), SHOW):
    po = idata.posterior[v].values.ravel(); pr = _pri[_map.get(v,v)].values
    lo = min(np.percentile(po,0.5), np.percentile(pr,2))
    hi = max(np.percentile(po,99.5), np.percentile(pr,98))
    g = np.linspace(lo,hi,240); dpr,dpo = kde1d(pr,g), kde1d(po,g)
    ax.fill_between(g,dpr/dpr.max(),color=STEEL,alpha=.30,label="prior")
    ax.fill_between(g,dpo/dpo.max(),color=BLUE,alpha=.55,label="posterior")
    ax.plot(np.percentile(po,[3,97]),[-.06,-.06],color=BLUE,lw=2.5,solid_capstyle="butt")
    if USE_SYNTHETIC:
        tv = dict(zip(PNAMES,THETA_TRUE)); tv["k_slow"]=min(THETA_TRUE[7],THETA_TRUE[8])
        ax.axvline(tv[v],color=INK,ls="--",lw=1.6,label="truth")
    ax.set_title(v,fontsize=9); ax.set_yticks([]); ax.set_ylim(-.14,1.15); ax.tick_params(labelsize=7)
axes.ravel()[0].legend(fontsize=6.5,loc="upper right")
fig.suptitle("Prior vs posterior",fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
PAIR = ["G","phi_crit","K_I","h"]
L = np.log(np.column_stack([idata.posterior[v].values.ravel() for v in PAIR]))
n = len(PAIR); fig,ax = plt.subplots(n,n,figsize=(7.2,7.0))
for i in range(n):
    for j in range(n):
        a = ax[i,j]
        if i == j:
            g = np.linspace(L[:,i].min(), L[:,i].max(), 200)
            a.fill_between(g, kde1d(L[:,i],g), color=BLUE, alpha=.5); a.set_yticks([])
        elif i > j:
            a.plot(L[:,j],L[:,i],".",ms=1.2,alpha=.25,color=BLUE)
        else:
            r = np.corrcoef(L[:,j],L[:,i])[0,1]
            a.text(.5,.5,f"r = {r:+.2f}",ha="center",va="center",transform=a.transAxes,
                   fontsize=11 if abs(r)>.4 else 9, color=INK if abs(r)>.4 else STEEL)
            a.set_xticks([]); a.set_yticks([])
        if i == n-1: a.set_xlabel(f"log {PAIR[j]}",fontsize=8)
        if j == 0 and i != 0: a.set_ylabel(f"log {PAIR[i]}",fontsize=8)
        a.tick_params(labelsize=6)
fig.suptitle("Joint posterior structure (log scale)",fontsize=10)
plt.tight_layout(); plt.show()

The pair plot should be read for structure as well as spread. A tight diagonal ridge means that two
parameters trade off against each other and indicates a need for reparametrisation. A wide but round
marginal means that a parameter is poorly constrained, which calls for more or better-placed data.
The two patterns have different causes and different remedies.

---
## 9 · Posterior predictive: the updated forecast

The result of interest is a posterior predictive distribution with credible intervals, not a point
estimate: for a given set of conditions, the expected yield and its uncertainty.

In [ ]:
KEYS=["mu_max","Xmax","phi_crit","P_d","K_I","h","eps","k_slow","k_fast","G","k_deg"]
def post_draws(idat, n=300, rng=RNG):
    f={v:idat.posterior[v].values.ravel() for v in KEYS}
    i=rng.choice(f["G"].size, size=min(n,f["G"].size), replace=False)
    return np.column_stack([f[k][i] for k in KEYS])

TH_P = post_draws(idata,300); Ppost,Xpost=[],[]
for th in TH_P:
    o=simulate(th,DOSES,t_fine)
    if o is not None: Ppost.append(o["P"]); Xpost.append(o["X"])
Ppost,Xpost=np.array(Ppost),np.array(Xpost)
a_post = np.exp(idata.posterior["log_alpha"].values.reshape(-1,N_GEL)).mean(0)

fig,ax=plt.subplots(2,len(DOSES),figsize=(3.1*len(DOSES),5.4),sharex=True)
for j,d in enumerate(DOSES):
    q=np.percentile(uM_to_mgL(Ppost[:,j,:])*a_post[GEL_ID[j]],[3,25,50,75,97],axis=0)
    ax[0,j].fill_between(t_fine,q[0],q[4],alpha=.15,color=BLUE)
    ax[0,j].fill_between(t_fine,q[1],q[3],alpha=.30,color=BLUE)
    ax[0,j].plot(t_fine,q[2],color=BLUE,lw=1.8)
    ax[0,j].plot(TOBS,CP_obs[j],"o",ms=5,color=INK,zorder=5)
    ax[0,j].set_title(f"{uM_to_mM(d):.3g} mM IPTG",fontsize=12)
    qo=np.percentile(Xpost[:,j,:],[3,50,97],axis=0)
    ax[1,j].fill_between(t_fine,qo[0],qo[2],alpha=.2,color=SKY)
    ax[1,j].plot(t_fine,qo[1],color=SKY,lw=1.8)
    ax[1,j].plot(TOBS,X_obs[j],"o",ms=5,color=INK,zorder=5)
    ax[1,j].set_xlabel("time (min)")
ax[0,0].set_ylabel("CP (mg L$^{-1}$)"); ax[1,0].set_ylabel("biomass (gDCW L$^{-1}$)")
plt.tight_layout(); plt.show()

In [ ]:
Ydp=[]
for th in TH_P[:200]:
    o=simulate(th,dscan,np.array([0.,T_END]))
    if o is not None: Ydp.append(yield_mgL(o["P"][:,-1],o["X"][:,-1]))
Ydp=np.array(Ydp); qp=np.percentile(Ydp,[5,50,95],axis=0)

fig,ax=plt.subplots(figsize=(6.6,4.0))
dmM = uM_to_mM(dscan)
ax.fill_between(dmM,qd[0],qd[4],alpha=.16,color=STEEL,label="prior 90%")
ax.plot(dmM,qd[2],color=STEEL,ls="--",lw=1.6,label="prior median")
ax.fill_between(dmM,qp[0],qp[2],alpha=.30,color=MID,label="posterior 90%")
ax.plot(dmM,qp[1],color=MID,lw=2.2,label="posterior median")
for d in DOSES: ax.axvline(uM_to_mM(d),color=INK,lw=.6,alpha=.30)
ax.axvspan(uM_to_mM(DOSES.max()), dmM[-1], color=INK, alpha=.06)
ax.text(np.sqrt(uM_to_mM(DOSES.max())*dmM[-1]), ax.get_ylim()[1]*0.06,
        "extrapolation\n(no data here)", ha="center", fontsize=7, color=INK)
ax.set(xscale="log",xlabel="IPTG (mM)",ylabel=f"CP yield at {T_END/60:.0f} h (mg/L)",
       title="Design forecast")
ax.legend(fontsize=7.5); plt.tight_layout(); plt.show()

wp,wq=np.median(qd[4]-qd[0]),np.median(qp[2]-qp[0]); k=int(np.argmax(qp[1]))
print(f"Median 90% width: prior {wp:.0f} -> posterior {wq:.0f} mg/L ({100*(1-wq/wp):.0f}% reduction)")
print(f"Posterior-optimal IPTG {uM_to_mM(dscan[k]):.3g} mM -> {qp[1][k]:.0f} mg/L "
      f"(90% CI {qp[0][k]:.0f}-{qp[2][k]:.0f})")

# Is the optimum actually resolved, or is the plateau flat within uncertainty?
flat = dmM[qp[1] >= 0.95*qp[1][k]]
print(f"Doses within 5% of the optimum: {flat.min():.3g}-{flat.max():.3g} mM "
      f"-> {'a real optimum' if flat.max()/flat.min() < 5 else 'FLAT: do not report a single optimal dose'}")

---
## 10 · Held-out validation

The model is refitted without one dose (150 µM, index 3 of `DOSES`) and used to predict it. If the
observed data fall inside the predictive interval, the yield prediction is supported.

In [ ]:
HOLDOUT = 3
keep = np.array([i for i in range(len(DOSES)) if i != HOLDOUT])
_D,_X,_CP,_L,_G = DOSES,X_obs,CP_obs,LOG_CP,GEL_ID
DOSES,X_obs,CP_obs,LOG_CP,GEL_ID = _D[keep],_X[keep],_CP[keep],_L[keep],_G[keep]

model_ho = build_model()
with model_ho:
    idata_ho = pm.sample_smc(draws=800, chains=4, random_seed=99, progressbar=False)

DOSES,X_obs,CP_obs,LOG_CP,GEL_ID = _D,_X,_CP,_L,_G
print(f"Refit without {_D[HOLDOUT]:.0f} uM; now predicting it.")

In [ ]:
TH_H = post_draws(idata_ho,300); Ph,Xh=[],[]
for th in TH_H:
    o=simulate(th,np.array([_D[HOLDOUT]]),t_fine)
    if o is not None: Ph.append(uM_to_mgL(o["P"][0])); Xh.append(o["X"][0])
Ph,Xh=np.array(Ph),np.array(Xh)
ah=np.exp(idata_ho.posterior["log_alpha"].values.reshape(-1,max(N_GEL,1))).mean(0)
au=ah[min(_G[HOLDOUT],ah.size-1)]

fig,ax=plt.subplots(1,2,figsize=(9,3.4))
q=np.percentile(Ph*au,[3,25,50,75,97],axis=0)
ax[0].fill_between(t_fine,q[0],q[4],alpha=.16,color=MID,label="94% predictive")
ax[0].fill_between(t_fine,q[1],q[3],alpha=.30,color=MID)
ax[0].plot(t_fine,q[2],color=MID,lw=2,label="median")
ax[0].plot(TOBS,_CP[HOLDOUT],"o",ms=7,color=INK,zorder=5,label="held-out data")
ax[0].set(xlabel="time (min)",ylabel="CP (mg L$^{-1}$)",
          title=f"Held-out: {_D[HOLDOUT]:.0f} $\\mu$M (never fitted)")
ax[0].legend(fontsize=7)
qo=np.percentile(Xh,[3,50,97],axis=0)
ax[1].fill_between(t_fine,qo[0],qo[2],alpha=.2,color=SKY)
ax[1].plot(t_fine,qo[1],color=SKY,lw=2)
ax[1].plot(TOBS,_X[HOLDOUT],"o",ms=7,color=INK,zorder=5)
ax[1].set(xlabel="time (min)",ylabel="biomass (gDCW L$^{-1}$)",title="Held-out growth")
plt.tight_layout(); plt.show()

lo=np.interp(TOBS,t_fine,np.percentile(Ph*au,3,axis=0))
hi=np.interp(TOBS,t_fine,np.percentile(Ph*au,97,axis=0))
ins=((_CP[HOLDOUT]>=lo)&(_CP[HOLDOUT]<=hi))
cov = ins.sum()/ins.size
print(f"Inside 94% predictive interval: {ins.sum()}/{ins.size} timepoints "
      f"({100*cov:.0f}%, nominal 94%)")
if FAST_MODE:
    print("NOTE: FAST_MODE is on. Under-sampled posteriors give unreliable coverage --")
    print("      re-run with FAST_MODE=False before drawing any conclusion from this number.")
elif cov < 0.7:
    print("Well below nominal -> the model is overconfident on unseen conditions.")
    print("Widen priors, or add a Kennedy-O'Hagan discrepancy term, before trusting forecasts.")
else:
    print("Coverage consistent with nominal: the held-out forecast is substantiated.")

---
## 11 · Export for Module M2

The M1 posterior serves as the prior for M2 (folding and aggregation). The full joint draws are
exported rather than the marginals alone, because the parameters are correlated.

In [ ]:
draws = pd.DataFrame({v: idata.posterior[v].values.ravel() for v in KEYS})
draws.to_csv("M1_iptg_posterior_draws.csv", index=False)
logd = np.log(draws)
pd.DataFrame({"median":draws.median(),"log_mean":logd.mean(),"log_sd":logd.std(),
              "q05":draws.quantile(.05),"q95":draws.quantile(.95)}
             ).to_csv("M1_iptg_posterior_moments.csv")
try:
    idata.to_netcdf("M1_iptg_idata.nc")           # needs netCDF4 or h5netcdf
    print("wrote M1_iptg_posterior_draws.csv / _moments.csv / _idata.nc\n")
except Exception as e:
    print(f"(NetCDF export skipped: {type(e).__name__}. "
          "pip install h5netcdf if you want the full InferenceData object.)")
    print("wrote M1_iptg_posterior_draws.csv / _moments.csv\n")

cm=logd.corr(); print("Posterior correlations |r|>0.4:")
for i in range(len(KEYS)):
    for j in range(i+1,len(KEYS)):
        if abs(cm.iloc[i,j])>0.4: print(f"  {KEYS[i]:9s} <-> {KEYS[j]:9s}  r={cm.iloc[i,j]:+.2f}")
print("\nCarry correlated pairs into M2 as JOINT draws, never independent marginals.")

---
## 12 · References

1. Monod J. The growth of bacterial cultures. *Annu Rev Microbiol.* 1949;3:371–394.
2. Scott M, Gunderson CW, Mateescu EM, Zhang Z, Hwa T. Interdependence of cell growth and gene expression: origins and consequences. *Science.* 2010;330:1099–1102.
3. Klumpp S, Zhang Z, Hwa T. Growth rate-dependent global effects on gene expression in bacteria. *Cell.* 2009;139:1366–1375.
4. Weiße AY, Oyarzún DA, Danos V, Swain PS. Mechanistic links between cellular trade-offs, gene expression, and growth. *PNAS.* 2015;112:E1038–E1047.
5. Bremer H, Dennis PP. Modulation of chemical composition and other parameters of the cell at different exponential growth rates. *EcoSal Plus.* 2008;3.
6. Milo R, Jorgensen P, Moran U, Weber G, Springer M. BioNumbers — the database of key numbers in molecular and cell biology. *Nucleic Acids Res.* 2010;38:D750–D753.
7. Marbach A, Bettenbrock K. lac operon induction in *Escherichia coli*: systematic comparison of IPTG and TMG induction. *J Biotechnol.* 2012;157:82–88.
8. Ozbudak EM, Thattai M, Lim HN, Shraiman BI, van Oudenaarden A. Multistability in the lactose utilization network of *Escherichia coli*. *Nature.* 2004;427:737–740.
9. Lewis M, Chang G, Horton NC, et al. Crystal structure of the lactose operon repressor and its complexes with DNA and inducer. *Science.* 1996;271:1247–1254.
10. Yildirim N, Mackey MC. Feedback regulation in the lactose operon. *Biophys J.* 2003;84:2841–2851.
11. Studier FW, Moffatt BA. Use of bacteriophage T7 RNA polymerase to direct selective high-level expression of cloned genes. *J Mol Biol.* 1986;189:113–130.
12. Dubendorff JW, Studier FW. Controlling basal expression in an inducible T7 expression system by blocking the target T7 promoter with lac repressor. *J Mol Biol.* 1991;219:45–59.
13. Bernstein JA, Khodursky AB, Lin P-H, Lin-Chao S, Cohen SN. Global analysis of mRNA decay and abundance in *Escherichia coli*. *PNAS.* 2002;99:9697–9702.
14. Selinger DW, Saxena RM, Cheung KJ, Church GM, Rosenow C. Global RNA half-life analysis in *Escherichia coli*. *Genome Res.* 2003;13:216–223.
15. Iost I, Guillerez J, Dreyfus M. Bacteriophage T7 RNA polymerase travels far ahead of ribosomes in vivo. *J Bacteriol.* 1992;174:619–622.
16. Vogel U, Jensen KF. The RNA chain elongation rate in *Escherichia coli* depends on the growth rate. *J Bacteriol.* 1994;176:2807–2813.
17. Klumpp S, Hwa T. Growth-rate-dependent partitioning of RNA polymerases in bacteria. *PNAS.* 2008;105:20245–20250.
18. Maurizi MR. Proteases and protein degradation in *Escherichia coli*. *Experientia.* 1992;48:178–201.
19. Rocke DM, Durbin B. A model for measurement error for gene expression arrays. *J Comput Biol.* 2001;8:557–569.
20. Gelman A, Hill J. *Data Analysis Using Regression and Multilevel/Hierarchical Models.* CUP; 2007.
21. Kennedy MC, O'Hagan A. Bayesian calibration of computer models. *J R Stat Soc B.* 2001;63:425–464.
22. Del Moral P, Doucet A, Jasra A. Sequential Monte Carlo samplers. *J R Stat Soc B.* 2006;68:411–436.
23. Raue A, Kreutz C, Maiwald T, et al. Structural and practical identifiability analysis of partially observed dynamical models. *Bioinformatics.* 2009;25:1923–1929.
